In [2]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(os.getenv("DATABASE_URL"))
df = pd.read_sql("SELECT * FROM cleaned_listings", conn)
conn.close()

print(f"Rows: {len(df)}")
print(f"Date range: {df['created'].min()} to {df['created'].max()}")
print(f"\nNull rates:")
print(df.isnull().mean().sort_values(ascending=False).round(3))


/var/folders/k2/09c8j5sd6bbf32rnpwj0kk600000gn/T/ipykernel_89307/1969057383.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM cleaned_listings", conn)


Rows: 963
Date range: 2023-09-23 15:01:18 to 2026-06-08 20:23:48

Null rates:
location_state       0.033
id                   0.000
title                0.000
company              0.000
location_raw         0.000
location_city        0.000
description_clean    0.000
salary_min           0.000
salary_max           0.000
salary_midpoint      0.000
is_remote            0.000
created              0.000
redirect_url         0.000
category             0.000
keywords_matched     0.000
cleaned_at           0.000
dtype: float64


In [3]:
print("Salary stats:")
print(df[["salary_min", "salary_max", "salary_midpoint"]].describe())

print(f"\nListings with salary_min == 0: {(df['salary_min'] == 0).sum()}")
print(f"Listings with salary_min is null: {df['salary_min'].isnull().sum()}")

print(f"\nListings by year:")
print(df['created'].dt.year.value_counts().sort_index())


Salary stats:
          salary_min     salary_max  salary_midpoint
count     963.000000     963.000000        963.00000
mean   117649.206542  121827.446417     119738.32648
std     47593.643114   44841.567130      45763.21800
min         0.000000   36016.400000      36016.40000
25%     82987.765000   88000.000000      83427.77000
50%    107034.130000  110691.140000     107328.92000
75%    142725.020000  144031.115000     142725.02000
max    305346.620000  310000.000000     305346.62000

Listings with salary_min == 0: 4
Listings with salary_min is null: 0

Listings by year:
created
2023      3
2024      1
2025     11
2026    948
Name: count, dtype: int64


In [4]:
print("Top 20 titles:")
print(df['title'].value_counts().head(20))

print(f"\nUnique titles: {df['title'].nunique()}")


Top 20 titles:
title
Data Engineer                                                       217
Data Analyst                                                        178
Mental Health Therapist                                             121
Analytics Engineer                                                   50
AV Safety Engineering Analytics Engineer (GPSSC)                     47
Senior Analytics Engineer (Platform - Financial Analytics)           45
GCP Cloud Analytics Engineer - Sr Consultant                         18
Senior Enterprise Applications Engineer                               9
Data Analytics Engineer                                               8
Senior Data Analyst                                                   8
DBT Sr. PM CMMC Specialist                                            6
Licensed Clinical Social Worker                                       6
Senior Data Engineer                                                  5
DATA ENGINEER                              

## Dead End: `dbt` keyword pulling mental health listings

The `dbt` keyword search returned 154 non-tech listings because "DBT" also stands 
for Dialectical Behavior Therapy. This is a real data quality problem — 16% of the 
dataset is irrelevant.

**Fix for next ingestion run:** Filter out listings where the title or category 
contains therapy-related terms, or drop the bare `dbt` keyword search and rely on 
`data engineer` + `analytics engineer` to surface dbt-related roles organically.


In [6]:
# dbt keyword is pulling mental health listings — check the scope
therapy_titles = df[df['title'].str.contains('Therapist|Therapist|Social Worker|Clinical', case=False, na=False)]
print(f"Non-tech listings pulled in by 'dbt' keyword: {len(therapy_titles)}")
print(therapy_titles['title'].value_counts().head(10))


Non-tech listings pulled in by 'dbt' keyword: 154
title
Mental Health Therapist                                                    121
Licensed Clinical Social Worker                                              6
Child and Adolescent Therapist                                               4
Adult Masters Licensed Therapist - DBT Specialist and Generalist             3
Licensed Clinical Professional Counselor                                     3
DBT Therapist                                                                2
Licensed Clinical Marriage and Family Therapist                              2
Registered Nurse (RN) Clinical Data Analyst Cardiac Cath Full Time Days      1
LICSW Supervisor / Lead DBT Therapist                                        1
Social Worker II                                                             1
Name: count, dtype: int64


## Finding: Adzuna truncates descriptions at 500 characters

25th, 50th, and 75th percentiles are all exactly 500 chars, with a max of 500.
The API is cutting off descriptions — we're not getting full job postings.

**Impact on Week 3:** Chunking strategy is simplified — no listing will exceed 
the token limit, so one chunk per listing is fine. But retrieval quality may 
suffer since descriptions are incomplete.


In [7]:
df['description_length'] = df['description_clean'].str.len()

print("Description length stats (characters):")
print(df['description_length'].describe())

print(f"\nListings under 200 chars: {(df['description_length'] < 200).sum()}")
print(f"Listings over 3000 chars: {(df['description_length'] > 3000).sum()}")


Description length stats (characters):
count    963.000000
mean     498.444444
std       20.409370
min      170.000000
25%      500.000000
50%      500.000000
75%      500.000000
max      500.000000
Name: description_length, dtype: float64

Listings under 200 chars: 1
Listings over 3000 chars: 0


In [8]:
# Flag data quality issues
bad_salary = df[df['salary_min'] == 0]
print(f"Listings with salary_min = 0 (bad data):")
print(bad_salary[['title', 'company', 'salary_min', 'salary_max']])

stale = df[df['created'].dt.year < 2026]
print(f"\nStale listings (pre-2026): {len(stale)}")
print(stale[['title', 'company', 'created']].to_string())


Listings with salary_min = 0 (bad data):
                                                 title  \
814  Travel Social Work - LMSW - Licensed Master So...   
863  Local Contract Mammography Technologist - $90 ...   
864  Travel Mammography Technologist - $4,058 per week   
876  Travel Social Work - Licensed Clinical Social ...   

                                      company  salary_min  salary_max  
814                                 LanceSoft         0.0     89440.0  
863  Amergis Healthcare Staffing, Inc.-Allied         0.0    187200.0  
864  Amergis Healthcare Staffing, Inc.-Allied         0.0    211016.0  
876  Amergis Healthcare Staffing, Inc.-Allied         0.0    127504.0  

Stale listings (pre-2026): 15
                                                                                               title                                    company             created
23                                                                                     Data Engineer             

## Summary of EDA Findings

| Issue | Count | Impact |
|-------|-------|--------|
| Non-tech listings from `dbt` keyword | 154 | Drop `dbt` as a standalone keyword |
| Stale listings (pre-2026) | 15 | Low impact, but worth filtering on ingest |
| Bad salary (min = 0) | 4 | Flag or exclude from salary analysis |
| Descriptions truncated at 500 chars | ~963 | One chunk per listing is the right call |
| location_state missing | ~32 | Acceptable, city is sufficient for filtering |

**Clean dataset for embedding:** ~809 listings after removing non-tech noise.
